# 06 — Advanced Topics: PyTorch Integration, Autograd & Next Steps

In this final notebook we close the loop between tensor network theory and practical deep learning:
- Differentiating through tensor contractions with PyTorch autograd
- Training a tensor-parameterised regression model end-to-end
- Quantum-inspired ML: the Born Machine density model
- Where to go next

---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import tensorly as tl
import tntorch as tn
import matplotlib.pyplot as plt
from numpy.linalg import norm

torch.manual_seed(42)
np.random.seed(42)
print("All imports OK.")


## 1. Autograd Through Tensor Contractions

Tensor contractions are just sequences of matrix multiplications and reshapes—
PyTorch differentiates through them automatically.

The TT-cores are `nn.Parameter` objects; gradients flow back through every
contraction step via the chain rule:

$$\nabla_{G^{(k)}} \mathcal{L} = \frac{\partial \mathcal{L}}{\partial G^{(k)}}$$


In [ ]:
# Two TT-cores: G1(1,d,r), G2(r,d,1)  →  contract to vector of size d^2
d, r = 3, 4
G1 = nn.Parameter(torch.randn(1, d, r))
G2 = nn.Parameter(torch.randn(r, d, 1))
target = torch.ones(d * d) / (d * d)**0.5

optimizer = optim.Adam([G1, G2], lr=0.05)
losses = []

for step in range(200):
    optimizer.zero_grad()
    # Contract: (1,d,r) x (r,d,1) -> (d,d) -> flatten
    contracted = torch.einsum('1dr,rDs->dD', G1.squeeze(0), G2.squeeze(-1))
    output = contracted.reshape(-1)
    loss = ((output - target)**2).sum()
    loss.backward()      # autograd through contraction
    optimizer.step()
    losses.append(loss.item())

print(f"Initial loss: {losses[0]:.5f}")
print(f"Final loss:   {losses[-1]:.8f}")
print(f"G1 grad norm: {G1.grad.norm().item():.5f}")
print(f"G2 grad norm: {G2.grad.norm().item():.5f}")

plt.figure(figsize=(6, 3))
plt.semilogy(losses)
plt.xlabel('Step'); plt.ylabel('Loss (log)'); plt.title('Optimising TT cores via autograd')
plt.tight_layout(); plt.show()


## 2. tntorch: Gradient Descent on TT-Tensors

`tntorch` stores TT-cores as PyTorch tensors, making them compatible with
standard `optim` loops. Here we approximate a 3-D function on a grid.


In [ ]:
N = 20
xs = torch.linspace(-np.pi, np.pi, N)
x1, x2, x3 = torch.meshgrid(xs, xs, xs, indexing='ij')
T_target = torch.sin(x1) * torch.cos(x2) + x3**2   # (20,20,20)
print(f"Target shape: {T_target.shape}, norm: {T_target.norm().item():.3f}")

# Initial TT approximation at bond dim 4
T_tt_init = tn.Tensor(T_target.detach().clone(), ranks_tt=4)
T_recon_0 = T_tt_init.torch()
err0 = (T_target - T_recon_0).norm() / T_target.norm()
print(f"Initial TT error (bond=4): {err0.item():.5f}")
print(f"TT params: {sum(c.numel() for c in T_tt_init.cores)} vs dense: {T_target.numel()}")

# Fine-tune with gradient descent
cores_params = [nn.Parameter(c.clone()) for c in T_tt_init.cores]
opt_tt = optim.Adam(cores_params, lr=1e-2)

tt_losses = []
for step in range(100):
    opt_tt.zero_grad()
    T_approx = tn.Tensor(cores_params).torch()
    loss = (T_approx - T_target).pow(2).mean()
    loss.backward()
    opt_tt.step()
    tt_losses.append(loss.item())

print(f"\nFinal MSE after fine-tuning: {tt_losses[-1]:.7f}")

plt.figure(figsize=(6, 3))
plt.semilogy(tt_losses)
plt.xlabel('Step'); plt.ylabel('MSE (log)'); plt.title('TT function approximation (gradient)')
plt.tight_layout(); plt.show()


## 3. Born Machine: Quantum-Inspired Density Estimation

A **Born machine** stores a probability distribution as $p(\mathbf{x}) = |\psi(\mathbf{x})|^2 / Z$
where $\psi$ is an MPS amplitude.

Training maximises $\mathcal{L} = \mathbb{E}_{\mathbf{x} \sim \mathcal{D}}[\log p(\mathbf{x})]$.

The partition function $Z = \langle\Psi|\Psi\rangle$ is computed exactly via MPS norm.


In [ ]:
# Data: binary strings, distribution favours high popcount
all_strings = np.array([[int(b) for b in f'{i:04b}'] for i in range(16)], dtype=np.float32)
weights = np.array([sum(row) + 1 for row in all_strings], dtype=np.float32)
probs_true = weights / weights.sum()

np.random.seed(7)
indices = np.random.choice(16, size=500, p=probs_true)
data = torch.tensor(all_strings[indices])
print(f"Training data: {data.shape}, true probs: {probs_true.round(3)}")


class BornMPS(nn.Module):
    def __init__(self, L=4, bond_dim=4, phys_dim=2):
        super().__init__()
        self.L, self.phys_dim = L, phys_dim
        self.cores = nn.ParameterList()
        for k in range(L):
            r_l = 1 if k == 0 else bond_dim
            r_r = 1 if k == L-1 else bond_dim
            self.cores.append(nn.Parameter(torch.randn(r_l, phys_dim, r_r) * 0.1))

    def amplitude(self, x):
        batch = x.shape[0]
        state = torch.ones(batch, 1)
        for k in range(self.L):
            x_k = x[:, k].long()
            # Select physical slice per sample: core[k][:, x_k, :] -> (batch, r_l, r_r)
            selected = self.cores[k][:, x_k, :].permute(1, 0, 2)
            state = torch.einsum('bi,bir->br', state, selected)
        return state.squeeze(-1)

    def log_prob(self, x):
        amp = self.amplitude(x)
        log_psi2 = 2 * torch.log(amp.abs() + 1e-10)
        all_x = torch.tensor(all_strings)
        Z = (self.amplitude(all_x)**2).sum()
        return log_psi2 - torch.log(Z)

    def get_probs(self):
        with torch.no_grad():
            all_x = torch.tensor(all_strings)
            amp2 = self.amplitude(all_x)**2
            return (amp2 / amp2.sum()).numpy()


model_bm = BornMPS(L=4, bond_dim=4, phys_dim=2)
opt_bm = optim.Adam(model_bm.parameters(), lr=5e-3)
loader_bm = DataLoader(TensorDataset(data), batch_size=64, shuffle=True)

bm_losses = []
for epoch in range(80):
    ep_loss = 0.0
    for (xb,) in loader_bm:
        opt_bm.zero_grad()
        nll = -model_bm.log_prob(xb).mean()
        nll.backward(); opt_bm.step()
        ep_loss += nll.item()
    bm_losses.append(ep_loss / len(loader_bm))

print("\nBorn machine trained.")
probs_learned = model_bm.get_probs()
kl = np.sum(probs_true * np.log(probs_true / (probs_learned + 1e-10)))
print(f"KL(true || learned) = {kl:.4f}  (lower is better)")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(bm_losses); ax1.set_xlabel('Epoch'); ax1.set_ylabel('NLL'); ax1.set_title('Born Machine NLL')
patterns = [''.join(map(str, row.astype(int))) for row in all_strings]
x_pos = np.arange(16)
ax2.bar(x_pos - 0.2, probs_true, 0.4, label='True', alpha=0.7)
ax2.bar(x_pos + 0.2, probs_learned, 0.4, label='Learned', alpha=0.7)
ax2.set_xticks(x_pos); ax2.set_xticklabels(patterns, rotation=90, fontsize=7)
ax2.set_ylabel('Probability'); ax2.set_title('True vs Learned'); ax2.legend()
plt.tight_layout(); plt.show()


## 4. TT Regression — Full Training Loop

A Tensor Train weight tensor combined with a polynomial feature map gives
a compact but expressive regression model.


In [ ]:
torch.manual_seed(0)
n_samples, n_features, phi_dim = 300, 5, 3

X = torch.randn(n_samples, n_features)
y_true = (X[:,0]*X[:,1] + X[:,2]**2 - X[:,3]*X[:,4]).unsqueeze(1)
y_true = y_true + 0.1 * torch.randn_like(y_true)

loader_reg = DataLoader(TensorDataset(X, y_true), batch_size=32, shuffle=True)

def poly2(xi):
    return torch.stack([torch.ones_like(xi), xi, xi**2], dim=-1)  # (batch, 3)


class TTRegressor(nn.Module):
    def __init__(self, n_features=5, phi_dim=3, bond_dim=4):
        super().__init__()
        self.n = n_features
        self.cores = nn.ParameterList()
        for k in range(n_features):
            rl = 1 if k==0 else bond_dim
            rr = 1 if k==n_features-1 else bond_dim
            self.cores.append(nn.Parameter(torch.randn(rl, phi_dim, rr) * 0.05))

    def forward(self, x):
        state = torch.ones(x.shape[0], 1)
        for k in range(self.n):
            phi_k = poly2(x[:, k])   # (batch, phi_dim)
            state = torch.einsum('bL,Lpr,bp->br', state, self.cores[k], phi_k)
        return state.squeeze(-1)


model_reg = TTRegressor()
opt_reg = optim.Adam(model_reg.parameters(), lr=5e-3)
mse = nn.MSELoss()

reg_losses = []
for epoch in range(150):
    ep = 0.0
    for xb, yb in loader_reg:
        opt_reg.zero_grad()
        loss = mse(model_reg(xb).unsqueeze(1), yb)
        loss.backward(); opt_reg.step()
        ep += loss.item()
    reg_losses.append(ep / len(loader_reg))

with torch.no_grad():
    final_mse = mse(model_reg(X).unsqueeze(1), y_true).item()
print(f"TT Regressor final MSE: {final_mse:.5f}")
print(f"Parameters: {sum(p.numel() for p in model_reg.parameters())}")

plt.figure(figsize=(6, 3))
plt.semilogy(reg_losses)
plt.xlabel('Epoch'); plt.ylabel('MSE (log)'); plt.title('TT Regressor Training')
plt.tight_layout(); plt.show()


## 5. Library Ecosystem Map

```
                       Your ML Model
                           │
              ┌────────────┼────────────┐
              │            │            │
         [PyTorch]    [TensorLy]    [quimb]
         autograd     decompose     physics
              │            │            │
         [tntorch]  [opt_einsum]   [TeNPy]
         TT-autograd fast paths   DMRG/MPS
              │
          [tn4ml]
          TN Layers (JAX)
```


In [ ]:
# ── Verify all library imports ──
checks = {}
for lib, imp in [
    ('numpy',         'import numpy'),
    ('tensorly',      'import tensorly'),
    ('tntorch',       'import tntorch'),
    ('opt_einsum',    'import opt_einsum'),
    ('tensornetwork', 'import tensornetwork'),
    ('quimb',         'import quimb'),
    ('tenpy',         'import tenpy'),
    ('tn4ml',         'import tn4ml'),
    ('torch',         'import torch'),
    ('jax',           'import jax'),
]:
    try:
        exec(imp); checks[lib] = '✓'
    except ImportError as e:
        checks[lib] = f'✗ ({e})'

print("Library check in tensor environment:")
print("-" * 42)
for lib, status in checks.items():
    print(f"  {lib:<22} {status}")


## Summary & Roadmap Recap

| Notebook | Topic | Key Tools |
|----------|-------|-----------|
| 01 | Scalars → tensors, einsum, contractions, unfolding | numpy, opt_einsum, tensorly |
| 02 | SVD, CP, Tucker, Tensor Train decompositions | tensorly |
| 03 | MPS, TTN, graphical notation, SVD splitting, contraction order | tensornetwork, opt_einsum |
| 04 | TT-layer compression, feature maps, MPS classifier | torch, tensorly |
| 05 | Independent per-library blocks | all 7 libraries |
| 06 | Autograd through contractions, Born machine, TT regression | tntorch, torch |

**Research directions:**
- TN layer compression of Transformers / CNNs
- Born machines for generative modelling
- Variational quantum-inspired algorithms
- DMRG-style sweep optimisation as an alternative to backprop
- Tensor networks in reinforcement learning (value function compression)

Activate the environment with `conda activate tensor`.